## Automatic Differentiation, Gradient Descent & Optimization 

Using Automatic Differentiation, implement Gradient Descent for the following equation: 

### f(x, y) = x^4 + xy + y^4 + x^3

The goal is to find an approximation of x & y such that f is minimized. Pick a random starting point x & y and print out the update steps until x_hat & y_hat is within some L2 norm of the actual x_min & y_min. Note how many update steps is needed for convergence. 

Modify your gradient descent algorithm to include i) momentum & ii) adaptive learning rates. Report the differences between the vanilla gradient descent and the others. 

-----------------------------

#### Steps
1.  Create class for automatic differentiation
2.  Initialize values for x and y
3.  Initialize learning rate and L2 norm threshold of x_min & y_min
4.  Compute partial of the function with respect to x
5.	Subtract x with the learning rate multiplied by the partial of the function with respect to x
6.	Repeat step 4 & 5 with y
7.	Calculate gradient for the updated x and y (partial derivatives). Now you get x_hat and y_hat for first iteration
8.	Calculate the L2 norm of these gradients
9.	Repeat steps 3 to 8 until L2 norms of the gradients for x_hat and y_hat (partial derivatives) reaches initialized L2 norm threshold 
10.	Print number of iterations taken to converge
11. Modify the algorithm to include momentum & rmsprop

In [3]:
import numpy as np

#### Class for Automatic Differentiation

In [4]:
class Variable():
    def __init__(self, f, d = 0):
        self.f = f
        self.d = d

    def __repr__(self):
        return f"Variable(f = {self.f}, d = {self.d})"

    def __add__(self, other):
        return Variable(self.f + other.f, self.d + other.d)

    def __radd__(self, other):
        return Variable(other.f + self.f, other.d + self.d)

    def __sub__(self, other):
        return Variable(self.f - other.f, self.d - other.d)

    def __rsub__(self, other):
        return Variable(other.f - self.f, other.d - self.d)

    def __mul__(self, other):
        return Variable(self.f * other.f, other.f * self.d + self.f * other.d)

    def __rmul__(self, other):
        return Variable(other.f * self.f, self.f * other.d + other.f * self.d)

    def __truediv__(self, other):
        return Variable(self.f / other.f, (self.d * other.f - self.f * other.d) / other.f ** 2)

    def __rtruediv__(self, other):
        return Variable(other.f / self.f, (other.d * self.f - other.f * self.d) / self.f ** 2)

    def log(self):
        return Variable(np.log(self.f), 1 / self.f * self.d)

    def __pow__(self, number):
        assert isinstance(number, (int, float)), "only supporting int/float powers"
        return Variable(np.power(self.f, number), number * np.power(self.f, number - 1) * self.d)

#### Vanilla Gradient Descent

In [11]:
x = 3
y = 2
l2_norm_threshold = 1e-3
l_r = 0.01 # learning rate
iterations = 50000
    
for i in range(iterations):
    
    x_var = Variable(x, 1)
    y_var = Variable(y, 0)
    partial_x = (np.power(x_var, 4) + x_var*y_var + np.power(y_var, 4) + np.power(x_var, 3)).d
    
    x_var = Variable(x, 0)
    y_var = Variable(y, 1)
    partial_y = (np.power(x_var, 4) + x_var*y_var + np.power(y_var, 4) + np.power(x_var, 3)).d

    x -= (l_r * partial_x)
    
    y -= (l_r * partial_y)
    
    x_var = Variable(x, 1)
    y_var = Variable(y, 0)
    
    x_hat = (np.power(x_var, 4) + x_var*y_var + np.power(y_var, 4) + np.power(x_var, 3)).d
    
    x_var = Variable(x, 0)
    y_var = Variable(y, 1)
    
    y_hat = (np.power(x_var, 4) + x_var*y_var + np.power(y_var, 4) + np.power(x_var, 3)).d
    
    l2_norm = np.sqrt(np.power(x_hat, 2) + np.power(y_hat, 2))

    if l2_norm < l2_norm_threshold:
        print('Steps to Converge:', i)
        break

Steps to Converge: 451


#### With Momentum

In [12]:
x = 3
y = 2
p = 0.9 # friction
v_x = 0 # velocity (x)
v_y = 0 # velocity (y)
    
for j in range(iterations):
    
    x_var = Variable(x, 1)
    y_var = Variable(y, 0)
    partial_x = (np.power(x_var, 4) + x_var*y_var + np.power(y_var, 4) + np.power(x_var, 3)).d
    
    x_var = Variable(x, 0)
    y_var = Variable(y, 1)
    partial_y = (np.power(x_var, 4) + x_var*y_var + np.power(y_var, 4) + np.power(x_var, 3)).d

    # adding momentum
    
    v_x = (p * v_x) + partial_x
    x -= (l_r * v_x)
    
    v_y = (p * v_y) + partial_y
    y -= (l_r * v_y)
    
    x_var = Variable(x, 1)
    y_var = Variable(y, 0)
    
    x_hat = (np.power(x_var, 4) + x_var*y_var + np.power(y_var, 4) + np.power(x_var, 3)).d
    
    x_var = Variable(x, 0)
    y_var = Variable(y, 1)
    
    y_hat = (np.power(x_var, 4) + x_var*y_var + np.power(y_var, 4) + np.power(x_var, 3)).d
    
    l2_norm = np.sqrt(np.power(x_hat, 2) + np.power(y_hat, 2))

    if l2_norm < l2_norm_threshold:
        print('Steps to Converge:', j)
        break

Steps to Converge: 160


#### With Adaptive Learning Rate

In [13]:
x = 3
y = 2
e = 1e-20
p = 0.9
sq_grads_x = 0 # squared gradients (x)
sq_grads_y = 0 # squared gradients (y)
    
for k in range(iterations):
    
    x_var = Variable(x, 1)
    y_var = Variable(y, 0)
    partial_x = (np.power(x_var, 4) + x_var*y_var + np.power(y_var, 4) + np.power(x_var, 3)).d
    
    x_var = Variable(x, 0)
    y_var = Variable(y, 1)
    partial_y = (np.power(x_var, 4) + x_var*y_var + np.power(y_var, 4) + np.power(x_var, 3)).d

    # rmsprop
    
    sq_grads_x = p * sq_grads_x + (1 - p) * partial_x * partial_x
    x -= l_r * (partial_x / (np.sqrt(sq_grads_x) + e))
    
    sq_grads_y = p * sq_grads_y + (1 - p) * partial_y * partial_y
    y -= l_r * (partial_y / (np.sqrt(sq_grads_y) + e))
    
    x_var = Variable(x, 1)
    y_var = Variable(y, 0)
    
    x_hat = (np.power(x_var, 4) + x_var*y_var + np.power(y_var, 4) + np.power(x_var, 3)).d
    
    x_var = Variable(x, 0)
    y_var = Variable(y, 1)
    
    y_hat = (np.power(x_var, 4) + x_var*y_var + np.power(y_var, 4) + np.power(x_var, 3)).d
    
    l2_norm = np.sqrt(np.power(x_hat, 2) + np.power(y_hat, 2))

    if l2_norm < l2_norm_threshold:
        print('Steps to Converge:', k)
        break

Steps to Converge: 345


#### Final Report

In [23]:
print('Steps to Converge')
print('-------------------------')
print('Vanilla GD:', i, 'steps')
print('With Momentum:', j, 'steps')
print('With RMSProp:', k, 'steps')

Steps to Converge
-------------------------
Vanilla GD: 451 steps
With Momentum: 160 steps
With RMSProp: 345 steps


---------------------------------------
MSc, M. D. R. K. P. M. (2022, March 24). Deep Learning: Automatic Differentiation (Part 1). Marcus D. R. Klarqvist. https://mdrk.io/automatic-differentiation-4/